# Prediksi Harga Rumah California
Eksperimen regresi prediksi harga rumah menggunakan algoritma XGBoost.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn import metrics

## 2. Load Data

In [ ]:
data_housing = pd.read_csv('housing.csv')
data_housing.head()

## 3. Cek Missing Value

In [ ]:
print('Ukuran Dataset:', data_housing.shape)
print('\nMissing Value Tiap Kolom:')
print(data_housing.isnull().sum())

## 4. EDA (Grafik Korelasi)

In [ ]:
korelasi = data_housing.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(korelasi, cbar=True, square=True, fmt='.2f', annot=True, annot_kws={'size': 9}, cmap='Blues')
plt.title('Peta Korelasi Fitur terhadap Harga Rumah')
plt.tight_layout()
plt.show()

## 5. Pisahkan Fitur & Target

In [ ]:
X = data_housing.drop(columns='MedHouseVal', axis=1)
y = data_housing['MedHouseVal']

print('Total data X:', X.shape)
print('Total data y:', y.shape)

## 6. Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

print('Total Data :', X.shape[0])
print('Data Latih :', X_train.shape[0])
print('Data Uji   :', X_test.shape[0])

## 7. Training Model (XGBoost)

In [ ]:
model = XGBRegressor(random_state=2)
model.fit(X_train, y_train)

## 8. Evaluasi Model

In [ ]:
# Evaluasi data latih
prediksi_latih = model.predict(X_train)
r2_latih = metrics.r2_score(y_train, prediksi_latih)
mae_latih = metrics.mean_absolute_error(y_train, prediksi_latih)

# Evaluasi data uji
prediksi_uji = model.predict(X_test)
r2_uji = metrics.r2_score(y_test, prediksi_uji)
mae_uji = metrics.mean_absolute_error(y_test, prediksi_uji)

print('=== NILAI DATA LATIH ===')
print('R-squared Score :', round(r2_latih, 4))
print('MAE             :', round(mae_latih, 4))

print('\n=== NILAI DATA UJI ===')
print('R-squared Score :', round(r2_uji, 4))
print('MAE             :', round(mae_uji, 4))

## 9. Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#0284c7')
plt.title('Tingkat Pengaruh Fitur terhadap Harga Rumah')
plt.xlabel('Importance')
plt.ylabel('Fitur')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 10. Visualisasi Prediksi vs Asli

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, prediksi_uji, color='blue', alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--', linewidth=2)
plt.xlabel('Harga Asli (x $100.000)')
plt.ylabel('Harga Prediksi (x $100.000)')
plt.title('Perbandingan Harga Asli vs Prediksi Model')
plt.tight_layout()
plt.show()

## 11. Taksiran Harga Data Baru

In [ ]:
data_rumah_baru = (8.3252, 41.0, 6.984127, 1.023810, 322.0, 2.555556, 37.88, -122.23)

data_array = np.asarray(data_rumah_baru).reshape(1, -1)
taksiran_harga = model.predict(data_array)[0]

harga_dollar = taksiran_harga * 100000
harga_rupiah = harga_dollar * 16000

print('HASIL TAKSIRAN HARGA PROPERTI')
print(f'Skor Model     : {taksiran_harga:.3f}')
print(f'Taksiran Harga : ${harga_dollar:,.2f} USD')
print(f'Setara Rupiah  : Rp {harga_rupiah:,.2f}')